In [1]:
"""
Notebook 03: Data Aggregation
=============================
Load full 2023, clean, aggregate to hourly time series
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Setup
DATA_DIR = Path("../data/raw")
PROCESSE_DIR = Path("../data/processed")
PROCESSE_DIR.mkdir(parents=True, exist_ok=True)

# plt.style.use('default')
# sns.set_palette("husl")

In [2]:
def load_year_data(year: int = 2023, taxi_type: str = "yellow") -> pd.DataFrame:
    """
    Load all monthly parquet files for a given year and concatenate.
    Args:
        year: Year of data to load.
        taxi_type: 'yellow' or 'green'

    Returns:
        Concatenated DataFrame with all months.
    """

    # We only read the needed columns - save the computer memory
    columns_needed = [
        'tpep_pickup_datetime',
        'tpep_dropoff_datetime',
        'passenger_count',
        'trip_distance',
        'total_amount',
        'fare_amount',
        'PULocationID',
        'DOLocationID'
    ]

    monthly_dfs = []
    # tqdm is always import to get feedback from the process
    for month in tqdm(range(1, 13), desc=f"loading {year}"):
        filename = f"{taxi_type}_tripdata_{year}-{month:02d}.parquet"
        filepath = DATA_DIR / filename

        if not filepath.exists():
            print(f"Missing: {filename}")
            continue
        # Parquet is column-oriented, thus it is going to read the needed columns
        # Parquet has a huge advantage opposite to CSV
        df_month = pd.read_parquet(filepath, columns=columns_needed)
        monthly_dfs.append(df_month)

    df = pd.concat(monthly_dfs, ignore_index=True)
    print(f" Loaded {len(df):,} rows total") # more visible
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    # deep = True, If it isn't adjusted, than it won't calculate correectly
    # the strings columns 
    return df


In [3]:
%%time
df_full = load_year_data(2023)

loading 2023: 100%|██████████| 12/12 [00:02<00:00,  4.74it/s]


 Loaded 38,310,226 rows total
Memory usage: 2.45 GB
CPU times: total: 8.69 s
Wall time: 4.02 s


In [4]:
df_full.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,total_amount,fare_amount,PULocationID,DOLocationID
0,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,14.30,9.3,161,141
1,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,16.90,7.9,43,237
2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,34.90,14.9,48,238
3,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,20.85,12.1,138,7
4,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,19.68,11.4,107,79


In [5]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38310226 entries, 0 to 38310225
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   passenger_count        float64       
 3   trip_distance          float64       
 4   total_amount           float64       
 5   fare_amount            float64       
 6   PULocationID           int64         
 7   DOLocationID           int64         
dtypes: datetime64[us](2), float64(4), int64(2)
memory usage: 2.3 GB


In [6]:
def clean_taxi_data(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Clean NYC taxi data based on data quality audit findings.

    Removes:
        - Negative or zero total amount (refunds, errors)
        - Negative or zero trip distance
        - Negative trip duration (pickup > dropoff)
        - Trips longer then 12 hours (outliers)
        - Tips with passenger_count == 0 or Nan

    Args:
        df: Raw Dataframe
        verbose: Print cleaning statistics

    Returns:
        Cleaned Dataframe
    """

    initial_rows = len(df)
    df = df.copy()

    # 1. Calculate trip duration in minutes
    df['trip_duration_min'] = (
        df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
    ).dt.total_seconds() / 60

    # Define filters
    filters = {
        'positive_amount': df['total_amount'] > 0,
        'positive_distance': df['trip_distance'] > 0,
        'positive_duration': df['trip_duration_min'] > 0,
        'reasonable_duration': df['trip_duration_min'] <= 12 * 60, # max 12 hours
        'has_passengers': (df['passenger_count'].notna()) & df['passenger_count'] > 0,
        'valid_pickup_year': df['tpep_pickup_datetime'].dt.year == 2023
    }

    # 3. Apply filters cumulatively, track losses
    if verbose:
        print(f"Initial rows: {initial_rows:,}")
        print("-" * 50)

    combined_mask = pd.Series(True, index=df.index)
    for filter_name, filter_mask in filters.items():
        before = combined_mask.sum()
        combined_mask &= filter_mask
        after = combined_mask.sum()
        removed = before - after
        if verbose:
            print(f' {filter_name:25s}: removed {removed:>10,} rows')

    df_clean = df[combined_mask].copy()

    if verbose:
        print("-" * 50)
        print(f"Final rows: {len(df_clean):,}")
        print(f"Removed: {initial_rows - len(df_clean):,} ({(1 - len(df_clean)/initial_rows) * 100:.2f}%)")

    return df_clean

In [7]:
df_clean = clean_taxi_data(df_full)

Initial rows: 38,310,226
--------------------------------------------------
 positive_amount          : removed    382,882 rows
 positive_distance        : removed    728,205 rows
 positive_duration        : removed      3,262 rows
 reasonable_duration      : removed     28,770 rows
 has_passengers           : removed  1,583,150 rows
 valid_pickup_year        : removed         66 rows
--------------------------------------------------
Final rows: 35,583,891
Removed: 2,726,335 (7.12%)


In [8]:
print("=== Trip distance analysis ===")
print(f"Total rows: {len(df_full):,}")
print(f"Negative distance: {(df_full['trip_distance'] < 0).sum():,}")
print(f"Zero distance:     {(df_full['trip_distance'] == 0).sum():,}")
print(f"Positive distance: {(df_full['trip_distance'] > 0).sum():,}")
print()
print("=== Per month: zero distance count ===")
print(df_full.groupby(df_full['tpep_pickup_datetime'].dt.month)['trip_distance']
      .apply(lambda x: (x == 0).sum()))

=== Trip distance analysis ===
Total rows: 38,310,226
Negative distance: 0
Zero distance:     773,457
Positive distance: 37,536,769

=== Per month: zero distance count ===
tpep_pickup_datetime
1      45861
2      41175
3      48450
4      40235
5      46892
6      47472
7      49791
8      57427
9      98546
10    121689
11    102586
12     73333
Name: trip_distance, dtype: int64


In [9]:
print("=== Passenger count problems per month ===")
problems_per_month = df_full.groupby(df_full['tpep_pickup_datetime'].dt.month).apply(
    lambda x: pd.Series({
        'total': len(x),
        'zero_passenger': (x['passenger_count'] == 0).sum(),
        'nan_passenger': x['passenger_count'].isnull().sum(),
        'problem_pct': ((x['passenger_count'] == 0) | x['passenger_count'].isnull()).sum() / len(x) * 100
    })
)
print(problems_per_month)

=== Passenger count problems per month ===
                          total  zero_passenger  nan_passenger  problem_pct
tpep_pickup_datetime                                                       
1                     3066759.0         51164.0        71743.0     4.007716
2                     2914003.0         47278.0        76817.0     4.258575
3                     3403660.0         58364.0        87619.0     4.289001
4                     3288248.0         56950.0        90690.0     4.489929
5                     3513664.0         59725.0       101796.0     4.596939
6                     3307259.0         54231.0        99887.0     4.659992
7                     2907093.0         43732.0        85086.0     4.431162
8                     2824201.0         41898.0        87886.0     4.595424
9                     2846741.0         39136.0       140225.0     6.300573
10                    3522280.0         46634.0       154929.0     5.722515
11                    3339732.0         41443

In [14]:
print("=== Pickup years in the 2023 data ===")
year_counts = df_full['tpep_pickup_datetime'].dt.year.value_counts().sort_index()
print(year_counts)

=== Pickup years in the 2023 data ===
tpep_pickup_datetime
2001           6
2002          11
2003           6
2008          23
2009          15
2014           1
2022          36
2023    38310122
2024           6
Name: count, dtype: int64


In [15]:
df_long = df_full.copy()
df_long['duration_min'] = (
    df_long['tpep_dropoff_datetime'] - df_long['tpep_pickup_datetime']
).dt.total_seconds() / 60
print(df_long.nlargest(10, 'duration_min')[
    ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'duration_min', 'trip_distance', 'total_amount']
])

         tpep_pickup_datetime tpep_dropoff_datetime  duration_min  \
2127658   2023-01-23 11:21:51   2023-01-30 10:31:02  10029.183333   
15333535  2023-05-24 12:51:56   2023-05-31 11:19:03   9987.116667   
23763760  2023-08-16 08:19:21   2023-08-22 16:36:26   9137.083333   
22522817  2023-08-02 11:05:52   2023-08-08 17:20:46   9014.900000   
25058918  2023-08-31 08:33:05   2023-09-05 11:05:12   7352.116667   
20624865  2023-07-14 12:52:23   2023-07-19 15:21:13   7348.833333   
23771787  2023-08-16 10:56:52   2023-08-21 12:03:43   7266.850000   
36632962  2023-12-15 11:42:05   2023-12-20 11:30:24   7188.316667   
5337796   2023-02-23 14:59:05   2023-02-28 12:32:42   7053.616667   
24845732  2023-08-28 18:13:03   2023-09-02 10:57:12   6764.150000   

          trip_distance  total_amount  
2127658            0.00          4.50  
15333535           0.00          4.50  
23763760           0.00          4.50  
22522817           0.00          5.20  
25058918           0.00          4.50  
